# PHYS 132 Activity 3: White dwarfs in the Gaia 100 pc sample

In Lecture 1 we drew the Hertzsprung--Russell diagram of a random sample of 10,000 stars within 100 pc of the
Sun, with distances from the parallaxes measured by the Gaia satellite (Data Release 3; the sample was assembled
by Brendan Bowler for this course). Below the main sequence there is a second, sparser sequence of faint, blue
stars. In this activity you will measure their temperatures and sizes from the Gaia data, and find out what
holds them up.

For each star the file gives the parallax (in milliarcseconds), the apparent magnitudes in the three Gaia bands
($G$, which is broad, and the blue and red halves $G_{\rm BP}$ and $G_{\rm RP}$), and two quality indicators.
The first cell keeps the stars with precise parallaxes, and computes the colour $G_{\rm BP} - G_{\rm RP}$ and
the absolute magnitude $M_G$, the magnitude each star would have at 10 pc. Smaller magnitudes are brighter, and
a difference of 5 magnitudes is a factor of 100 in flux.

Run the first cell as it is. In each part after that, fill in the lines marked `...` and run the cell. The
questions at the end of each part are for your group to discuss and to answer in a sentence or two.

In [ ]:
# Run this cell first. It loads the Gaia sample and defines the constants and the Planck function, in cgs units.
import numpy as np
import matplotlib.pyplot as plt

d = np.genfromtxt("gaia_100pc_sample.csv", delimiter=",", names=True)
good = ((d["parallax_over_error"] > 20) & (d["ruwe"] < 1.4)
        & np.isfinite(d["phot_bp_mean_mag"]) & np.isfinite(d["phot_rp_mean_mag"]))
d = d[good]
colour = d["phot_bp_mean_mag"] - d["phot_rp_mean_mag"]           # BP - RP colour
M_G = d["phot_g_mean_mag"] + 5 * np.log10(d["parallax"]) - 10     # absolute G magnitude (parallax in mas)

h, c, k_B = 6.626e-27, 2.998e10, 1.381e-16
G, m_e, m_u = 6.674e-8, 9.109e-28, 1.661e-24
Msun, Rsun = 1.989e33, 6.957e10
T_sun, colour_sun, M_G_sun = 5772.0, 0.82, 4.67    # the Sun's temperature, BP-RP colour and absolute G magnitude
lam_BP, lam_RP, lam_G = 511e-7, 777e-7, 640e-7      # effective wavelengths of the three Gaia bands, cm

def B_lam(lam, T):
    """The Planck function per unit wavelength (Homework 2, problem 4, has it per unit frequency)."""
    return 2 * h * c**2 / lam**5 / np.expm1(h * c / (lam * k_B * T))

print(f"{len(d)} stars with good parallaxes and colours")

## 1. Finding the white dwarfs

The faint blue sequence lies below the line $M_G = 3.5\,(G_{\rm BP} - G_{\rm RP}) + 9.5$ on the diagram. Select
the stars below it (that is, with larger $M_G$) and bluer than $G_{\rm BP} - G_{\rm RP} = 1.5$.

1. How many white dwarfs are there in the sample, and what fraction of all the stars is that?
2. The bluest white dwarfs have the colours of main-sequence stars of a few solar masses, so their surfaces are
   at least as hot, yet they are about ten magnitudes fainter. What does that tell you about their size, before
   any calculation?

In [ ]:
wd = ...   # True for the white dwarfs

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.scatter(colour, M_G, s=1.5, c="0.6", label="all stars")
ax.scatter(colour[wd], M_G[wd], s=4, c="C0", label="white dwarfs")
cc = np.linspace(-0.5, 1.5, 10)
ax.plot(cc, 3.5 * cc + 9.5, "k--", lw=1)
ax.set_xlim(-0.6, 4.2); ax.set_ylim(17, -4)
ax.set_xlabel(r"$G_{\rm BP} - G_{\rm RP}$"); ax.set_ylabel(r"$M_G$")
ax.legend()
plt.show()
print(f"{wd.sum()} white dwarfs out of {len(wd)} stars")

## 2. Temperatures from colours

Treat each star as a blackbody. The colour measures the ratio of the fluxes in the blue and red bands, which we
approximate by the Planck function at each band's effective wavelength:
$$
G_{\rm BP} - G_{\rm RP} = -2.5\log_{10}\frac{B_\lambda(\lambda_{\rm BP}, T)}{B_\lambda(\lambda_{\rm RP}, T)}
+ \text{constant}.
$$
The constant depends on how the magnitude scales are defined, and the cell fixes it by requiring that a
blackbody at the Sun's temperature have the Sun's colour. You only need to write the first term. The cell then
tabulates the colour of a blackbody against temperature and reads off each white dwarf's temperature.

1. Does a hotter blackbody look bluer or redder by this measure, and why?
2. What range of surface temperatures do the white dwarfs have?
3. Two approximations go into this: what are they?

In [ ]:
def raw_colour(T):
    return ...   # -2.5 log10 of the ratio of B_lam at the BP and RP wavelengths

zp = colour_sun - raw_colour(T_sun)                 # the constant, fixed by the Sun
T_grid = np.logspace(np.log10(2500), np.log10(40000), 500)
colour_grid = raw_colour(T_grid) + zp
T_wd = np.interp(colour[wd], colour_grid[::-1], T_grid[::-1])   # colour falls as T rises, so reverse the grid

plt.semilogx(T_grid, colour_grid)
plt.xlabel("T (K)"); plt.ylabel(r"blackbody $G_{\rm BP} - G_{\rm RP}$")
plt.show()
print("white dwarf temperatures, 10th, 50th and 90th percentiles:", np.percentile(T_wd, [10, 50, 90]).round(-2), "K")

## 3. Radii

The flux a star sends us in the $G$ band is proportional to its surface area times the Planck function at the
band's wavelength, divided by the square of its distance. The absolute magnitude puts every star at the same
distance, so comparing a star with the Sun,
$$
10^{-0.4\,(M_G - M_{G,\odot})} = \left(\frac{R}{R_\odot}\right)^2 \frac{B_\lambda(\lambda_G, T)}{B_\lambda(\lambda_G, T_\odot)} .
$$
Solve this for $R/R_\odot$ for each white dwarf. The cell also applies your formula to main-sequence stars with
the Sun's colour, as a check.

1. What is the typical radius of a white dwarf, in solar radii and in Earth radii ($R_\oplus = 0.0092\,R_\odot$)?
2. Does the check on the Sun-like stars come out as it should?

In [ ]:
def radius(M_abs, T):
    return ...   # R / R_sun

R_wd = radius(M_G[wd], T_wd)
sunlike = ~wd & (np.abs(colour - colour_sun) < 0.05)
print(f"Sun-like main-sequence stars: median R = {np.median(radius(M_G[sunlike], T_sun)):.2f} R_sun")
print("white dwarf radii, 10th, 50th and 90th percentiles:", np.percentile(R_wd, [10, 50, 90]).round(4), "R_sun")

plt.hist(R_wd / 0.0092, bins=30)
plt.xlabel(r"$R/R_\oplus$"); plt.ylabel("number of white dwarfs")
plt.show()

## 4. What holds them up

The masses of white dwarfs are measured from binary orbits and from the widths of their spectral lines, and
most are close to $0.6\,M_\odot$. Take that mass and your median radius, and work out the mean density, the
electron density (with $\mu_e = 2$, as for helium, carbon or oxygen), and the Fermi momentum from the lecture,
$p_F = h(3n_e/8\pi)^{1/3}$. The cell then converts $p_F$ into a Fermi temperature, the kinetic energy at the top
of the Fermi sea divided by $k_B$, and compares three pressures: the average pressure the virial theorem asks
for (Section 5 of the lecture), the degeneracy pressure of nonrelativistic electrons at the mean density, and the
thermal pressure the electrons would have at $10^7$\,K, a typical temperature for the interior of a white dwarf.

1. Are the electrons degenerate? Are they relativistic?
2. Which pressure holds a white dwarf up?

In [ ]:
M_wd = 0.6 * Msun
R_med = np.median(R_wd) * Rsun

rho_mean = ...   # mean density, g cm^-3
n_e = ...        # electrons per cm^3
p_F = ...        # Fermi momentum

T_F = (np.sqrt((p_F * c)**2 + (m_e * c**2)**2) - m_e * c**2) / k_B
P_virial = 3 * G * M_wd**2 / (20 * np.pi * R_med**4)
P_degenerate = (1 / 20) * (3 / np.pi)**(2 / 3) * h**2 / m_e * n_e**(5 / 3)
P_thermal = n_e * k_B * 1e7
print(f"mean density {rho_mean:.2e} g/cm^3, n_e = {n_e:.2e} cm^-3, p_F/(m_e c) = {p_F/(m_e*c):.2f}, T_F = {T_F:.2e} K")
print(f"pressures (dyn/cm^2): virial {P_virial:.2e}, degenerate {P_degenerate:.2e}, thermal at 1e7 K {P_thermal:.2e}")

## 5. If you have time: the spread in radii

In the lecture we estimated $R \approx 0.012\,R_\odot\,(M/0.6\,M_\odot)^{-1/3}$.

1. How does that compare with your median radius?
2. If all of the spread in your radii (say from the 10th to the 90th percentile) came from a spread in mass,
   what range of masses would it imply?
3. White dwarf masses measured directly cluster within about $0.1\,M_\odot$ of $0.6\,M_\odot$. What does that
   suggest about where the spread in your radii comes from?

In [ ]:
M_range = ...   # masses, in M_sun, for the 10th and 90th percentile radii
print(M_range)